# Escalabilidade — resultados de execução

Lê os logs em `results/escal_forte_*.txt` (escalabilidade forte: tamanho fixo, threads variando) e `results/escal_fraca_*.txt` (escalabilidade fraca: tamanho cresce com as threads), um par por política de scheduling (`static`/`dynamic`/`guided`). Plota tempo, speedup e eficiência vs. número de threads, comparando as três políticas.

In [ ]:
import re
import statistics as st
from pathlib import Path

import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")

FIELD_PATTERNS = {
    "threads_usadas": r"threads_usadas=(\d+)",
    "tempo": r"^tempo=([\d.]+)",
}


def parse_runs(path):
    """Extrai um dict por execucao do ./train encontrada no arquivo
    (o log e' a saida do terminal colada crua -- prompts/comandos de
    shell no meio sao ignorados, o parser so procura os campos)."""
    text = Path(path).read_text()
    starts = [m.start() for m in re.finditer(r"threads_usadas=\d+", text)]
    blocks = [text[s:e] for s, e in zip(starts, starts[1:] + [len(text)])]
    runs = []
    for block in blocks:
        run = {}
        for field, pattern in FIELD_PATTERNS.items():
            m = re.search(pattern, block, re.MULTILINE)
            if m:
                run[field] = int(m.group(1)) if field == "threads_usadas" else float(m.group(1))
        if "threads_usadas" in run and "tempo" in run:
            runs.append(run)
    return runs


def group_by_threads(runs):
    """threads_usadas -> lista de tempos (s), um por execucao repetida."""
    groups = {}
    for r in runs:
        groups.setdefault(r["threads_usadas"], []).append(r["tempo"])
    return dict(sorted(groups.items()))

In [ ]:
POLICIES = {
    "static": "tab:blue",
    "dynamic": "tab:orange",
    "guided": "tab:green",
}

STRONG_FILES = {
    "static": "escal_forte_result.txt",
    "dynamic": "escal_forte_dynamic.txt",
    "guided": "escal_forte_guided.txt",
}
WEAK_FILES = {
    "static": "escal_fraca_result.txt",
    "dynamic": "escal_fraca_dynamic.txt",
    "guided": "escal_fraca_guided.txt",
}

strong_groups = {policy: group_by_threads(parse_runs(RESULTS_DIR / fname))
                  for policy, fname in STRONG_FILES.items()}
weak_groups = {policy: group_by_threads(parse_runs(RESULTS_DIR / fname))
               for policy, fname in WEAK_FILES.items()}

for policy, groups in strong_groups.items():
    print(f"forte/{policy}: " + ", ".join(f"p={p} (n={len(v)})" for p, v in groups.items()))
for policy, groups in weak_groups.items():
    print(f"fraca/{policy}: " + ", ".join(f"p={p} (n={len(v)})" for p, v in groups.items()))

## Escalabilidade forte

Tamanho do problema fixo (`--ref-size 5000 --batch 32 --iters 3500`), número de threads variando. T1 = mediana das execuções com `threads_usadas=1` (a mesma nos três arquivos, já que a política de schedule não tem efeito com uma única thread). Speedup = T1/Tp, eficiência = speedup/p.

In [ ]:
fig, (ax_t, ax_s, ax_e) = plt.subplots(1, 3, figsize=(16, 4.5))

for policy, groups in strong_groups.items():
    threads = sorted(groups.keys())
    medians = [st.median(groups[p]) for p in threads]
    T1 = medians[threads.index(1)]
    speedup = [T1 / m for m in medians]
    efficiency = [s / p for s, p in zip(speedup, threads)]
    color = POLICIES[policy]

    ax_t.plot(threads, medians, "o-", label=policy, color=color)
    ax_s.plot(threads, speedup, "o-", label=policy, color=color)
    ax_e.plot(threads, efficiency, "o-", label=policy, color=color)

ax_t.set_xlabel("threads"); ax_t.set_ylabel("tempo mediano (s)"); ax_t.set_title("Tempo"); ax_t.legend()

max_p = max(t for g in strong_groups.values() for t in g)
ax_s.plot([1, max_p], [1, max_p], "k--", alpha=0.4, label="ideal (y=p)")
ax_s.set_xlabel("threads"); ax_s.set_ylabel("speedup (T1/Tp)"); ax_s.set_title("Speedup"); ax_s.legend()

ax_e.axhline(1.0, color="k", linestyle="--", alpha=0.4, label="ideal (100%)")
ax_e.set_xlabel("threads"); ax_e.set_ylabel("eficiência (speedup/p)"); ax_e.set_title("Eficiência"); ax_e.legend()
ax_e.set_ylim(0, 1.1)

fig.suptitle("Escalabilidade forte — static vs. dynamic vs. guided")
plt.tight_layout()
plt.show()

### Tabela — escalabilidade forte

In [ ]:
print(f"{'policy':<9} {'p':>3} {'mediana(s)':>11} {'stdev':>8} {'speedup':>8} {'eficiência':>11}")
for policy, groups in strong_groups.items():
    threads = sorted(groups.keys())
    T1 = st.median(groups[1])
    for p in threads:
        vals = groups[p]
        med = st.median(vals)
        sd = st.stdev(vals) if len(vals) > 1 else 0.0
        speedup = T1 / med
        eff = speedup / p
        print(f"{policy:<9} {p:>3} {med:>11.3f} {sd:>8.3f} {speedup:>8.2f} {eff:>10.1%}")

## Escalabilidade fraca

Tamanho do problema cresce com as threads (`--batch 32*p`, `--ref-size 5000 --iters 3500` fixos). Eficiência aqui é T1/Tp direto (sem dividir por p, já que o trabalho por thread já é mantido ~constante).

In [ ]:
fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(11, 4.5))

for policy, groups in weak_groups.items():
    threads = sorted(groups.keys())
    medians = [st.median(groups[p]) for p in threads]
    T1 = medians[threads.index(1)]
    efficiency = [T1 / m for m in medians]
    color = POLICIES[policy]

    ax_t.plot(threads, medians, "o-", label=policy, color=color)
    ax_e.plot(threads, efficiency, "o-", label=policy, color=color)

ax_t.set_xlabel("threads"); ax_t.set_ylabel("tempo mediano (s)"); ax_t.set_title("Tempo"); ax_t.legend()

ax_e.axhline(1.0, color="k", linestyle="--", alpha=0.4, label="ideal (100%)")
ax_e.set_xlabel("threads"); ax_e.set_ylabel("eficiência (T1/Tp)"); ax_e.set_title("Eficiência"); ax_e.legend()
ax_e.set_ylim(0, 1.1)

fig.suptitle("Escalabilidade fraca — static vs. dynamic vs. guided")
plt.tight_layout()
plt.show()

### Tabela — escalabilidade fraca

In [ ]:
print(f"{'policy':<9} {'p':>3} {'batch':>6} {'mediana(s)':>11} {'stdev':>8} {'eficiência':>11}")
for policy, groups in weak_groups.items():
    threads = sorted(groups.keys())
    T1 = st.median(groups[1])
    for p in threads:
        vals = groups[p]
        med = st.median(vals)
        sd = st.stdev(vals) if len(vals) > 1 else 0.0
        eff = T1 / med
        print(f"{policy:<9} {p:>3} {32*p:>6} {med:>11.3f} {sd:>8.3f} {eff:>10.1%}")